In [ ]:
%%bash 

curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
#%pip install -U crewai crewai_tools 'crewai[tools, litellm]' unstructured

##### Setup and Run ollama locally

```bash

#
curl -fsSL https://ollama.com/install.sh | sh
#
sudo systemctl status ollama
sudo systemctl start ollama
sudo systemctl stop ollama

#
docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama

#
ollama list

#
/set parameter num_ctx 32768

#
# models : sudo ls -l /usr/share/ollama/.ollama/models
#
ollama pull llama3.2:1b-instruct-q8_0
ollama run llama3.2:1b-instruct-q8_0 
ollama stop llama3.2:1b-instruct-q8_0

ollama pull deepseek-r1:1.5b
ollama run deepseek-r1:1.5b
ollama stop deepseek-r1:1.5b

curl -X POST http://localhost:11434/api/generate -d '{"model": "llama3.2:1b-instruct-q8_0", "prompt":"Why is the sky blue?","options":{"num_ctx":4096} }'


#
# embeddings
#
ollama pull all-minilm
ollama run all-minilm
curl http://localhost:11434/v1/embeddings -d '{"model": "all-minilm", "prompt": "Why is the sky blue?"}'
curl http://localhost:11434/api/embeddings -d '{"model": "all-minilm", "prompt": "Why is the sky blue?"}'


# http://localhost:11434
# http://localhost:11434/v1/models

curl http://localhost:11434/api/ps

curl http://localhost:11434/api/embed -d '{
  "model": "all-minilm",
  "input": "Why is the sky blue?"
}'

curl http://localhost:11434/api/show -d '{
  "model": "llama3.2:1b-instruct-q8_0"
}'


ollama rm llama3.2:1b-instruct-q8_0
# /usr/share/ollama/.ollama/models
curl -X DELETE http://localhost:11434/api/delete -d '{ "model": "llama3.2:1b-instruct-q8_0" }'
```

#### Seup Docker Model Runner Locally

```bash
sudo -S apt-get update
sudo -S apt-get install docker-model-plugin

docker model reinstall-runner --backend vllm --gpu cuda

docker model pull ai/gpt-oss:20B
docker model run --detach ai/gpt-oss:20B
docker model stop ai/gpt-oss:20B

docker model pull ai/smollm2
docker model run --detach ai/smollm2
docker model stop ai/smollm2

# Pull a small, efficient model (good for getting started)
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2:1B-Q8_0

# Pull a larger, more capable model
docker model pull ai/llama3.2:3B-Q4_K_M

# Pull a code-specialized model
docker model pull ai/codellama:7B-Q4_K_M

docker model pull ai/llama3.2
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2
docker model stop ai/llama3.2

# List all downloaded models with sizes
docker model list

# Show detailed information about a model
docker model inspect ai/llama3.2:1B-Q8_0

# Remove a model to free disk space
docker model rm ai/llama3.2:1B-Q8_0

# Remove all unused models
docker model prune

docker model uninstall-runner --models
docker model uninstall-runner --images

http://localhost:12434/engines/v1/models

http://vmware-ubuntu.sandbox.net:12434
# The model is now serving on a local endpoint
# Default: http://localhost:12434/v1

#
curl http://localhost:12434/engines/v1/embeddings \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/qwen3-embedding",
    "input": "A dog is an animal"
  }'
  
curl http://localhost:12434/api/chat \
  -H "Content-Type: application/json" \
  -d '{"model": "ai/smollm2", "messages": [{"role": "user", "content": "Hello!"}]}'

# Send a chat completion request using curl
curl http://localhost:12434/engines/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/llama3.2:1B-Q8_0",
    "messages": [
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Explain Docker volumes in 3 sentences."}
    ],
    "temperature": 0.7,
    "max_tokens": 256
  }'
```

```yaml
# ./docker-compose.yaml
services:
  app:
    image: my-app
    models:
      llm:
        endpoint_var: AI_MODEL_URL
        model_var: AI_MODEL_NAME
      embedding-model:
        endpoint_var: EMBEDDING_URL
        model_var: EMBEDDING_NAME

models:
  dev_model:
    model: ai/model
    context_size: 4096
    runtime_flags:
  llm:
    model: ai/smollm2
    context_size: 4096
    runtime_flags:
      - "--temp 0.1"            # Temperature
      - "--top-p 0.9"           # Top-p sampling
      - "--verbose"             # Set verbosity level to infinity
      - "--verbose-prompt"      # Print a verbose prompt before generation
      - "--log-prefix"          # Enable prefix in log messages
      - "--log-timestamps"      # Enable timestamps in log messages
      - "--log-colors"          # Enable colored logging
  embedding-model:
    model: ai/all-minilm
    context_size: 2048
    runtime_flags:
      - "--embeddings"
```

In [ ]:
#Setup config dir

%mkdir -p agentic-ai/crewai/config

In [ ]:
# %pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/agents.yaml
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest
    developments in {topic}. You find the most relevant information and
    present it clearly.
EOF


In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research about {topic}. Use web search to find current,
    credible information. The current year is 2026.
  expected_output: >
    A markdown report with clear sections: key trends, notable tools or companies,
    and implications. Aim for 800–1200 words. No fenced code blocks around the whole document.
  agent: researcher
  output_file: output/report.md
EOF

In [ ]:
# src/latest_ai_flow/crews/content_crew/content_crew.py

from typing import List

from crewai import LLM, Agent, Crew, Process, Task
from crewai.agents.agent_builder.base_agent import BaseAgent
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool

llm = LLM(
   model="ollama/llama3.2:1b-instruct-q8_0",
   base_url="http://localhost:11434"
)

@CrewBase
class ResearchCrew:
  """Single-agent research crew used inside the Flow."""

  agents: List[BaseAgent]
  tasks: List[Task]

  agents_config = "agentic-ai/crewai/config/agents.yaml"
  tasks_config = "agentic-ai/crewai/config/tasks.yaml"

  @agent
  def researcher(self) -> Agent:
    return Agent(
      config=self.agents_config["researcher"],  # type: ignore[index]
      verbose=True,
      tools=[SerperDevTool()],
      llm=llm
    )

  @task
  def research_task(self) -> Task:
    return Task(
      config=self.tasks_config["research_task"],  # type: ignore[index]
    )

  @crew
  def crew(self) -> Crew:
    return Crew(
      agents=self.agents,
      tasks=self.tasks,
      process=Process.sequential,
      verbose=True,
    )

In [ ]:
# src/latest_ai_flow/main.py

from pydantic import BaseModel
from crewai.flow import Flow, listen, start
#from latest_ai_flow.crews.content_crew.content_crew import ResearchCrew


class ResearchFlowState(BaseModel):
  topic: str = ""
  report: str = ""


class LatestAiFlow(Flow[ResearchFlowState]):
  @start()
  def prepare_topic(self, crewai_trigger_payload: dict | None = None):
    if crewai_trigger_payload:
      self.state.topic = crewai_trigger_payload.get("topic", "AI Agents")
    else:
      self.state.topic = "AI Agents"
    print(f"Topic: {self.state.topic}")

  @listen(prepare_topic)
  def run_research(self):
    result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
    self.state.report = result.raw
    print("Research crew finished.")

  @listen(run_research)
  def summarize(self):
    print("Report path: output/report.md")


def kickoff():
  LatestAiFlow().kickoff()


def plot():
  LatestAiFlow().plot()


#if __name__ == "__main__":
#  kickoff()

In [ ]:
plot()

In [ ]:
kickoff()

In [ ]:
import os
from crewai import LLM
from crewai import Agent, Task, Crew
from langchain_openai.chat_models import ChatOpenAI
from langchain_ollama.llms import OllamaLLM
from crewai import Crew, Process

ollama_llm = ChatOpenAI(
    model="llama3.2:1b-instruct-q8_0",  # Replace with your desired model
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="http://localhost:11434/v1",
    temperature=0.7  # Adjust creativity level,
)

response = ollama_llm.invoke("Hello")
response.pretty_print()

#langchain llm 
ollama_llm = OllamaLLM(
    model="llama3.2:1b-instruct-q8_0",
    temperature=0.7,
    num_predict=256,
    base_url="http://localhost:11434",
    # other params...
)

In [ ]:
import os
from crewai import LLM, Crew, Process, Agent, Task
from langchain_openai.chat_models import ChatOpenAI
from langchain_ollama.llms import OllamaLLM

# Before (LiteLLM):
# llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434)

# After (OpenAI-compatible mode, no LiteLLM needed):
ollama_llm = LLM(
  model="openai/llama3.2:1b-instruct-q8_0",
  base_url="http://localhost:11434/v1",
  temperature=0.8,
  api_key="ollama"  # Ollama doesn't require a real API key
)

# docker_llm = LLM(
#     model="openai/ai/llama3.2:1B-Q8_0",
#     base_url="http://localhost:12434/engines/v1"
# )




# Define your agents
researcher = Agent(
    role='Senior Research Analyst',
    goal='Discover new insights',
    backstory="""You're an expert at finding interesting information""",
    llm=ollama_llm,
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role='Content Writer',
    goal='Write engaging content',
    backstory="""You're a talented writer who simplifies complex information""",
    llm=ollama_llm,
    verbose=True
)

# Create tasks
research_task = Task(
    description='Find interesting facts about AI in healthcare',
    expected_output="A clear, concise summary about AI in healthcare",
    agent=researcher
)

write_task = Task(
    description='Write a short blog post about AI in healthcare',
    expected_output="Write a short blog post about AI in healthcare",
    agent=writer
)

# Form the crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

# Execute the crew's tasks
result = crew.kickoff()

print("Here's the result:")
print(result)

In [ ]:
import os
from crewai import Agent, Task, Crew
# Importing crewAI tools
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

# Set up API keys
os.environ["SERPER_API_KEY"] = "Your Key" # serper.dev API key
os.environ["OPENAI_API_KEY"] = "Your Key"

# Instantiate tools
docs_tool = DirectoryReadTool(directory='./blog-posts')
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

# Create agents
researcher = Agent(
    role='Market Research Analyst',
    goal='Provide up-to-date market analysis of the AI industry',
    backstory='An expert analyst with a keen eye for market trends.',
    tools=[search_tool, web_rag_tool],
    verbose=True
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging blog posts about the AI industry',
    backstory='A skilled writer with a passion for technology.',
    tools=[docs_tool, file_tool],
    verbose=True
)

# Define tasks
research = Task(
    description='Research the latest trends in the AI industry and provide a summary.',
    expected_output='A summary of the top 3 trending developments in the AI industry with a unique perspective on their significance.',
    agent=researcher
)

write = Task(
    description='Write an engaging blog post about the AI industry, based on the research analyst's summary. Draw inspiration from the latest blog posts in the directory.',
    expected_output='A 4-paragraph blog post formatted in markdown with engaging, informative, and accessible content, avoiding complex jargon.',
    agent=writer,
    output_file='blog-posts/new_post.md'  # The final blog post will be saved here
)

# Assemble a crew with planning enabled
crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    verbose=True,
    planning=True,  # Enable planning feature
)

# Execute tasks
crew.kickoff()

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
#model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

In [ ]:

from crewai import LLM
from pydantic import BaseModel

class Dog(BaseModel):
    name: str
    age: int
    breed: str

ollama_llm = LLM(
  model="ollama/llama3.2:1b-instruct-q8_0",
  base_url="http://localhost:11434/v1",
  temperature=0.8,
  response_format=Dog,
  api_key="ollama"  # Ollama doesn't require a real API key
)

response = ollama_llm.call(
    "Analyze the following messages and return the name, age, and breed. "
    "Meet Kona! She is 3 years old and is a black german shepherd."
)
print(response)

In [19]:
import asyncio
from crewai import LLM

async def stream_async():
    llm = LLM(
        model="ollama/llama3.2:1b-instruct-q8_0",
        base_url="http://localhost:11434/v1",
        temperature=0.8,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )
    response = await llm.acall("Write a short story about AI")
    print(response)

#asyncio.run(stream_async())





In [ ]:
await stream_async()

Output()

In [ ]:
import nest_asyncio
nest_asyncio.apply()
asyncio.run(stream_async()) # Now will work


In [ ]:
#
loop = asyncio.get_running_loop()
loop.create_task(stream_async())

In [ ]:
# To execute a new event loop without interfering with the current one, you can run your async code in a background thread using ThreadPoolExecutor.
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor() as executor:
    result = executor.submit(lambda: asyncio.run(stream_async())).result()

In [18]:
#
import asyncio
from crewai import LLM

async def main():
    llm = LLM(
        model="ollama/llama3.2:1b-instruct-q8_0",
        base_url="http://localhost:11434/v1",
        temperature=0.8,
        response_format=Dog,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )

    # Single async call
    response = await llm.acall("What is the capital of France?")
    print(response)

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction
from crewai.rag.chromadb.types import ChromaEmbeddingFunctionWrapper
from typing import Literal, cast

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig



set_rag_config(ChromaDBConfig(
    embedding_function=cast(
        ChromaEmbeddingFunctionWrapper,
        OllamaEmbeddingFunction(
            model_name="all-minilm",
            url="http://localhost:11434/api/embeddings"
        ),
    )
))

chromadb_client = get_rag_client()

# Qdrant
#from crewai.rag.qdrant.config import QdrantConfig
#set_rag_config(QdrantConfig())
#qdrant_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client
client.create_collection(collection_name="docs")
client.add_documents(
    collection_name="docs",
    documents=[{"id": "1", "content": "CrewAI enables collaborative AI agents."}],
)
results = client.search(collection_name="docs", query="collaborative agents", limit=3)

clear_rag_config()  # optional reset

In [7]:
from crewai_tools import PDFSearchTool

# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config

tool = PDFSearchTool(
    pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config={
        "embedding_model": {
            # Supported providers: "openai", "azure", "google-generativeai", "google-vertex",
            # "voyageai", "cohere", "huggingface", "jina", "sentence-transformer",
            # "text2vec", "ollama", "openclip", "instructor", "onnx", "roboflow", "watsonx", "custom"
            "provider": "openai",  # or: "google-generativeai", "cohere", "ollama", ...
            "config": {
                # Model identifier for the chosen provider. "model" will be auto-mapped to "model_name" internally.
                "model": "all-minilm",
                # Optional: API key. If omitted, the tool will use provider-specific env vars
                # (e.g., OPENAI_API_KEY or EMBEDDINGS_OPENAI_API_KEY for OpenAI).
                # "api_key": "sk-...",

                # Provider-specific examples:
                # --- Google Generative AI ---
                # (Set provider="google-generativeai" above)
                # "model_name": "gemini-embedding-001",
                # "task_type": "RETRIEVAL_DOCUMENT",
                # "title": "Embeddings",

                # --- Cohere ---
                # (Set provider="cohere" above)
                # "model": "embed-english-v3.0",

                # --- Ollama (local) ---
                # (Set provider="ollama" above)
                # "model": "nomic-embed-text",
            },
        },
        "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        # For ChromaDB: pass "settings" (chromadb.config.Settings) or rely on defaults.
                        # Example (uncomment and import):
                        # from chromadb.config import Settings
                        # "settings": Settings(
                        #     persist_directory="/content/chroma",
                        #     allow_reset=True,
                        #     is_persistent=True,
                        # ),

                        # For Qdrant: pass "vectors_config" (qdrant_client.models.VectorParams).
                        # Example (uncomment and import):
                        # from qdrant_client.models import VectorParams, Distance
                        # "vectors_config": VectorParams(size=384, distance=Distance.COSINE),

                        # Note: collection name is controlled by the tool (default: "rag_tool_collection"), not set here.
                    }
        },
    }
)


CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [5]:
tool.run("What are the skills ?")

"Relevant Content:\n\nimproved release frequency from monthly to bi-weekly, ensuring on-time delivery against stringent regulatory \n\ndeadlines. \n\n \n\n• Led Data Engineering Chapter for IB Markets Regulatory Reporting, governing 5+ regulatory \n\nframeworks (MIFID-II, RTS23, EUBR/UKBR, SSD, SFTR) — overseeing ingestion, reconciliation, and distribution \n\nof deal-store data for ~10 million+ daily trade & transaction records with full audit trail and zero regulatory \n\nbreach incidents across a 3-year delivery window. \n\n \n\n\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama', # Required by SDK but ignored locally
)

In [6]:
response = client.embeddings.create(
    model="all-minilm",
    input="This is a test sentence to generate embeddings."
)

# Access the vector
embedding_vector = response.data[0].embedding
print(embedding_vector)

[-0.019671116024255753, -0.014650529250502586, 0.060604989528656006, 0.06876762956380844, 0.01930004544556141, 0.0498303547501564, 0.003419275628402829, -0.06220618635416031, 0.0024810717441141605, -0.039921052753925323, 0.08755725622177124, -0.07096315920352936, 0.07399594038724899, 0.0028060676995664835, -0.030089033767580986, 0.025123313069343567, 0.08793304115533829, 0.005675734486430883, -0.05189467966556549, -4.27056584157981e-05, 0.03608176112174988, 0.06750448048114777, 0.06351044028997421, -0.04577000066637993, 0.02164362370967865, -0.027181070297956467, -0.04468441754579544, 0.05580121651291847, 0.11913752555847168, -0.024639733135700226, 0.08204834908246994, -0.007067996542900801, -0.030826520174741745, 0.062036409974098206, 0.06521482765674591, 0.04746629297733307, 0.04139960929751396, 0.04639935493469238, 0.00646931491792202, 0.01957232691347599, 0.017449168488383293, -0.03657972067594528, 0.0523429736495018, 0.07779139280319214, 0.01872655190527439, -0.038051072508096695,